In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import os 
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC']
plt.rcParams['axes.unicode_minus'] = False
import os 

from sklearn.metrics import normalized_mutual_info_score as NMI
from tqdm import tqdm
from esl import one_hot_encode_sequences, ESLClassifier, ESLPSCClassifier, leave_one_out_eval

## 展示ESL ESL_PSC的结果

选取top50特征

### ZWS demo

In [2]:
def pipe(species_id):
    # species_id = 'Sorex_araneus'
    dir_path = f'../phd_thesis/103_leave/{species_id}'
    feature_df = pd.read_csv(os.path.join(dir_path, 'df_feature.csv'), index_col = 0)
    meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'),index_col = 0)
    summary_df = pd.read_csv(os.path.join(dir_path, 'df_summary.csv'),index_col = 0)
    assert (meta_df.index == feature_df.index).all()
    assert (summary_df.index == feature_df.columns).all()

    all_species = meta_df.index.to_list()
    train_species = meta_df.index.to_list()
    train_species.remove(species_id)

    train_label   = meta_df.loc[train_species, 'label'].values

    # 全量特征（含 species_id），用于最终全量预测
    def get_top_n(group, n=100):
        return group.nlargest(n, 'NMI')

    gene_list = []
    for ele in summary_df.index: 
        gene = ele.split('_')[0]
        gene_list.append(gene)
    summary_df.loc[:,'gene'] = gene_list
    summary_top100 = summary_df.groupby('gene', group_keys=False).apply(get_top_n, n=100)

    all_feature = pd.read_parquet('/home/rsun@ZHANGroup.local/hqy_new/data_717/feature_df/data_raw.parquet')
    #train_feature = all_feature.loc[summary_top100.index,train_species].T 
    
    '''ESL 预测'''
    train_id = []
    for ele in all_species:
        if ele == species_id:
            train_id.append(False)
        else:
            train_id.append(True)
    train_id = np.array(train_id)
    res = []
    # one-hot 编码
    X, feature_names, group_ids, position_names, gene_names = one_hot_encode_sequences(all_feature.loc[summary_top100.index,all_species].T)
    X_train = X[train_id,:]
    X_test = X[~train_id,:]
    print(X_train.shape, X_test.shape)
    y = train_label 

    # 训练
    clf = ESLClassifier(lambda1=0.03, lambda2=0.01,tol = 1e-4, max_iter = 100)
    clf.fit(X_train, y, group_ids, feature_names=feature_names,
            position_names=position_names, gene_names=gene_names)

    # 预测
    y_pred = clf.predict(X_test)           # 0/1 标签
    proba = clf.predict_proba(X_test)      # shape (n, 2), 第1列为正类概率
    sps = clf.decision_function(X_test)    # Sequence Prediction Score
    print(f"eval {meta_df.loc[species_id,'label'],meta_df.loc[species_id,'species_chinese']}")
    print(f'ESL prediction label {y_pred}, eco proba {proba}')
    res = [species_id,meta_df.loc[species_id,'label'],meta_df.loc[species_id,'species_chinese'],y_pred, proba]
    
    '''ESL-PSC'''
    # 留出测试物种
    test_species = species_id
    seq_df = all_feature.loc[summary_top100.index,all_species].T
    train_sp = [s for s in seq_df.index if s != test_species]

    # 训练
    clf = ESLPSCClassifier(
        lambda1_list=[0.03],   # lambda1 搜索范围
        lambda2_list=[0.01],   # lambda2 搜索范围
        top_pct=0.3,                  # 保留 top 30% 的模型
        n_alternates=2,               # 每个 order 最多取2种配对
        order_col="order_chinese_new",
        label_col="label",
    )
    clf.fit(seq_df.loc[train_sp], meta_df.loc[train_sp],max_iter = 100,tol = 1e-4)

    # 单物种预测
    y_pred, proba, sps = clf.predict_species(seq_df, test_species)
    res.append(y_pred)
    res.append(proba)
    print(f'ESL-PSC prediction label {y_pred}, eco proba {proba}')
    print('==================================================================')
    return res


In [4]:
# pipe( species_id = 'Sorex_araneus')

In [7]:
import os
import multiprocessing

# if __name__ == '__main__':
data_dir = '../phd_thesis/103_leave/'
species_list = os.listdir(data_dir)

print(f"Found {len(species_list)} species. Starting parallel processing...")

all_results = [] # 用于存储所有物种的结果

# 使用进程池
with multiprocessing.Pool(processes=52) as pool:
    results_from_pool = pool.map(pipe, species_list)
    
# 聚合结果
for species, res in zip(species_list, results_from_pool):
    # res 是 pipe 返回的列表，例如 [(y_pred, proba)]
    # 你可以选择将其展平，或者保留结构
    all_results.extend(res) 
    
    # 如果需要记录哪个结果属于哪个物种，建议这样存：
    # for item in res:
    #     all_results.append({'species': species, 'pred': item[0], 'proba': item[1]})

print(f"Processing complete. Total records collected: {len(all_results)}")
    
    # 后续可以将 all_results 转换为 DataFrame 或其他格式进行分析
    # import pandas as pd
    # df_results = pd.DataFrame(all_results, columns=['y_pred', 'proba']) 

Found 104 species. Starting parallel processing...


/tmp/ipykernel_1576809/659598544.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_top100 = summary_df.groupby('gene', group_keys=False).apply(get_top_n, n=100)
/tmp/ipykernel_1576809/659598544.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_top100 = summary_df.groupby('gene', group_keys=False).apply(get_top_n, n=100)
/tmp/ipykernel_1576809/659598544.py:25: DeprecationWarning: DataFr

(103, 51123) (1, 51123)
(103, 51139) (1, 51139)
(103, 51341) (1, 51341)
(103, 49123) (1, 49123)
(103, 51238) (1, 51238)
(103, 51327) (1, 51327)
(103, 51214) (1, 51214)
(103, 51302) (103, 51282)(1, 51302) 
(1, 51282)
(103, 51157) (1, 51157)
(103, 51157) (1, 51157)(103, 49355)
 (1, 49355)
(103, 51194) (1, 51194)
(103, 51301) (1, 51301)
(103, 50970) (1, 50970)(103, 51191) 
(1, 51191)
(103, 51305) (1, 51305)
(103, 51324) (103, 51287)(1, 51324) 
(1, 51287)
(103, 51154) (1, 51154)
(103, 49072) (1, 49072)
(103, 51225) (1, 51225)
(103, 51193) (1, 51193)(103, 51131)
 (103, 49194)(1, 51131) 
(1, 49194)
(103, 51128) (1, 51128)
(103, 51364) (1, 51364)
(103, 51322) (1, 51322)
(103, 51067) (1, 51067)
(103, 51328) (1, 51328)
(103, 51216) (1, 51216)
(103, 51175) (1, 51175)
(103, 49504) (1, 49504)
(103, 51170)(103, 51238) (1, 51170)(103, 51266) (1, 51238)

 (1, 51266)
(103, 51264) (1, 51264)
(103, 49315) (1, 49315)
(103, 51253) (1, 51253)
(103, 51285) (1, 51285)
(103, 51195) (1, 51195)
(103, 49001) (1,

/tmp/ipykernel_1576809/659598544.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_top100 = summary_df.groupby('gene', group_keys=False).apply(get_top_n, n=100)


(103, 52723) (1, 52723)
eval (1.0, '猪尾鼠')
ESL prediction label [0], eco proba [[0.60663396 0.39336604]]
ESL-PSC prediction label 1, eco proba 0.6749240764592347


...

(103, 51126) (1, 51126)
(103, 51104) (1, 51104)
ESL-PSC prediction label 0, eco proba 0.33090458713266324
ESL-PSC prediction label 0, eco proba 0.24760052204489785
(103, 51203) (1, 51203)
(103, 51133) (1, 51133)
(103, 51138) (1, 51138)


/tmp/ipykernel_1576809/659598544.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_top100 = summary_df.groupby('gene', group_keys=False).apply(get_top_n, n=100)


(103, 51147) (1, 51147)


/tmp/ipykernel_1576809/659598544.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_top100 = summary_df.groupby('gene', group_keys=False).apply(get_top_n, n=100)


eval (0.0, '绿猴')
ESL prediction label [0], eco proba [[0.74497454 0.25502546]]
(103, 51225) (1, 51225)
eval (1.0, '太平洋短吻海豚')
ESL prediction label [1], eco proba [[0.3123932 0.6876068]]
(103, 49192) (1, 49192)
eval (0.0, '倭黑猩猩')
ESL prediction label [0], eco proba [[0.72683375 0.27316625]]
(103, 51136) (1, 51136)
eval (0.0, '小家鼠')
ESL prediction label [0], eco proba [[0.70279696 0.29720304]]
eval (0.0, '智人')
ESL prediction label [0], eco proba [[0.74413432 0.25586568]]
eval (1.0, '帕拉斯獒蝠')
ESL prediction label [1], eco proba [[0.33088058 0.66911942]]
eval (0.0, '藏酋猴')
ESL prediction label [0], eco proba [[0.72835023 0.27164977]]
eval (1.0, '大卫鼠耳蝠')
ESL prediction label [1], eco proba [[0.35980832 0.64019168]]
eval (0.0, '恒河猴')
ESL prediction label [0], eco proba [[0.74843928 0.25156072]]
eval (0.0, '玻利维亚松鼠猴')
ESL prediction label [0], eco proba [[0.73986227 0.26013773]]
eval (0.0, '黑叶猴')
ESL prediction label [0], eco proba [[0.74036746 0.25963254]]
eval (1.0, '大棕蝠')
ESL prediction label 

In [11]:
new_res = []
for i in range(104):
    tmp = all_results[7*i:7*i + 7]
    idx = tmp[0]
    label = tmp[1]
    idx_cn = tmp[2]
    pred_1 = tmp[3][0]
    pred_prob_1 = tmp[4][0][1]
    pred_2 = tmp[5]
    pred_prob_2 = tmp[6]
    new_res.append([idx, label, idx_cn, pred_1, pred_prob_1, pred_2, pred_prob_2])
df = pd.DataFrame(new_res)
df.columns = ['species','label','species_cn','esl_label','esl_eco_prob','esl_psc_label','esl_psc_eco_prob']
df.index = df['species'].values 
df.to_csv('esl_pred_v2.csv')

In [10]:
df

,species,label,species_cn,esl_label,esl_eco_prob,esl_psc_label,esl_psc_eco_prob
Bos_mutus,Bos_mutus,0.0,野牦牛,0,0.279495,0,0.266982
Ochotona_princeps,Ochotona_princeps,0.0,美洲鼠兔,0,0.317306,0,0.419894
Pan_troglodytes,Pan_troglodytes,0.0,黑猩猩,0,0.258839,0,0.253939


In [12]:
idx_1 = df['esl_label'].values != df['label'].values 
print(idx_1.sum())

idx_2= df['esl_psc_label'].values != df['label'].values 
print(idx_2.sum())

print(df['species_cn'][idx_1])
print(df['species_cn'][idx_2])

8
5
Pteropus_giganteus     印度狐蝠
Pteropus_alecto         黑狐蝠
ZWS                     猪尾鼠
Echinops_telfairi      小马岛猬
Pteropus_vampyrus     马来大狐蝠
Sorex_araneus          普通鼩鼱
shrew_mole               鼩鼹
Physeter_catodon        抹香鲸
Name: species_cn, dtype: object
Panthera_tigris_altaica     东北虎
Echinops_telfairi          小马岛猬
Rousettus_aegyptiacus      埃及果蝠
Sorex_araneus              普通鼩鼱
shrew_mole                   鼩鼹
Name: species_cn, dtype: object


In [13]:
df.loc[idx_1]

,species,label,species_cn,esl_label,esl_eco_prob,esl_psc_label,esl_psc_eco_prob
Pteropus_giganteus,Pteropus_giganteus,0.0,印度狐蝠,1,0.622166,0,0.315561
Pteropus_alecto,Pteropus_alecto,0.0,黑狐蝠,1,0.591863,0,0.282686
ZWS,ZWS,1.0,猪尾鼠,0,0.393366,1,0.561395
Echinops_telfairi,Echinops_telfairi,1.0,小马岛猬,0,0.361939,0,0.451282
Pteropus_vampyrus,Pteropus_vampyrus,0.0,马来大狐蝠,1,0.560660,0,0.257612
Sorex_araneus,Sorex_araneus,1.0,普通鼩鼱,0,0.451013,0,0.386666
shrew_mole,shrew_mole,1.0,鼩鼹,0,0.356892,0,0.400550
Physeter_catodon,Physeter_catodon,1.0,抹香鲸,0,0.489635,1,0.585904


### 准备全部数据

In [68]:
leave_species_id = meta_df.index 

In [69]:
for _, species_id in enumerate(leave_species_id):
    print(f'{_} {species_id} =====================================')
    # species_id = 'ZWS'

    '''创建save_dir'''
    save_dir = f'103_leave/{species_id}'
    os.makedirs(save_dir,exist_ok=True)

    ### 加速计算，使用预计算好的104 物种的summary, 只保留前50000 NMI feature 参与计算
    summary_df_pre = pd.read_csv('/home/rsun@ZHANGroup.local/hqy_new/paper_result/summary_df.csv', index_col=0)
    summary_df_pre = summary_df_pre.sort_values(by='score', ascending=False)
    pre_feature = summary_df_pre.index[:20000]

    feature_df = pd.read_parquet('/home/rsun@ZHANGroup.local/hqy_new/data_717/feature_df/data_raw.parquet').T
    feature_df = feature_df.loc[:,pre_feature]
    meta_df = pd.read_csv('/home/rsun@ZHANGroup.local/hqy_new/metadata/metadata_1.csv',index_col= 0)

    train_id = meta_df['label'].values != 2
    train_species = meta_df.index[train_id]

    feature_df = feature_df.loc[train_species,:]
    meta_df = meta_df.loc[train_species, :]

    train_species = list(train_species)
    train_species.remove(species_id)

    test_feature = feature_df.loc[species_id,:].values

    feature_df = feature_df.loc[train_species,:]
    meta_df = meta_df.loc[train_species,:]
    print(f"""train species num {feature_df.shape[0]} \n 
    ======== pos number {(meta_df['label'].values == 1).sum()} \n
    ======== neg number {(meta_df['label'].values == 0 ).sum()}""")

    '''perform feature selection'''

    score_map_4 = {4: 1, 3:0.95, 2: 0.9, 1:0.1, 0:0}
    score_map_5 = {5: 1, 4: 0.9, 3: 0.75, 2: 0.5, 1:0.1,0: 0}

    ### 计算nmi
    feature_num = feature_df.shape[1]
    assert (meta_df.index == feature_df.index).all()

    y = meta_df['label'].values
    nmi_list = []


    for i in tqdm(range(feature_num)):
        y_pred = feature_df.iloc[:,i].values 
        nmi = NMI(y, y_pred)
        nmi_list.append(nmi)

    ### 获取eco mutation
    eco_idx = meta_df['label'].values == 1 
    eco_feature = feature_df.loc[eco_idx,:]
    eco_mutation = eco_feature.mode(axis = 0).iloc[0,:].values  

    ### 获取eco mutation cover 数目  4支或5支
    cover_res = {}
    eco_meta = meta_df.loc[eco_idx, :]

    clade_num = 5
    for key in ['啮齿目','真盲缺目','非洲兽目','翼手目','鲸目']:
        
        if key not in eco_meta['order_chinese_new'].values:
            cover_res[key] = np.zeros(len(eco_mutation))
            clade_num = 4 ## 至多只有1个回声谱系消失
        else:
            idx = eco_meta['order_chinese_new'] == key 
            cover_res[key] = (eco_feature.loc[idx,:] == eco_mutation).sum(axis = 0) / idx.sum()
    cover_df = pd.DataFrame(cover_res)
    cover_df.loc[:,'NMI'] = nmi_list 
    cover_df.loc[:,'Eco_Mode'] = eco_mutation 
    cover_df.loc[:,'eco_cover'] = (cover_df.loc[:,['啮齿目','真盲缺目','非洲兽目','翼手目','鲸目']] > 0).sum(1)
    if clade_num == 4:
        cover_df.loc[:,'cover_score'] = cover_df.loc[:,'eco_cover'].map(score_map_4)
    if clade_num == 5: 
        cover_df.loc[:,'cover_score'] = cover_df.loc[:,'eco_cover'].map(score_map_5)

    cover_df.loc[:,'score'] = cover_df.loc[:,'cover_score'] * cover_df.loc[:,'NMI']
    cover_df = cover_df.sort_values(by='score', ascending=False)

    '''save data'''
    cover_df = cover_df.iloc[:10000,:]
    feature_df = pd.read_parquet('/home/rsun@ZHANGroup.local/hqy_new/data_717/feature_df/data_raw.parquet').T
    feature_df = feature_df.loc[:,cover_df.index]
        
    meta_df = pd.read_csv('/home/rsun@ZHANGroup.local/hqy_new/metadata/metadata_1.csv',index_col= 0)
    train_id = meta_df['label'].values != 2
    train_species = meta_df.index[train_id]

    feature_df = feature_df.loc[train_species,:]
    meta_df = meta_df.loc[train_species, :]

    cover_df.to_csv(os.path.join(save_dir, 'df_summary.csv'))
    feature_df.to_csv(os.path.join(save_dir, 'df_feature.csv'))
    meta_df.to_csv(os.path.join(save_dir, 'df_meta.csv'))




0 Chrysochloris_asiatica =====================================
train species num 103 
 
    ======== pos number 30 

    ======== neg number 73


100%|██████████| 20000/20000 [00:25<00:00, 797.82it/s]


1 Balaenoptera_acutorostrata_scammoni =====================================
train species num 103 
 
    ======== pos number 30 

    ======== neg number 73


100%|██████████| 20000/20000 [00:24<00:00, 800.02it/s]


2 Balaenoptera_musculus =====================================
train species num 103 
 
    ======== pos number 30 

    ======== neg number 73


...

100%|██████████| 20000/20000 [00:25<00:00, 799.03it/s]


102 shrew_mole =====================================
train species num 103 
 
    ======== pos number 29 

    ======== neg number 74


100%|██████████| 20000/20000 [00:25<00:00, 793.26it/s]


103 ZWS =====================================
train species num 103 
 
    ======== pos number 29 

    ======== neg number 74


100%|██████████| 20000/20000 [00:25<00:00, 791.95it/s]


In [20]:
summary_df_pre = pd.read_csv('/home/rsun@ZHANGroup.local/hqy_new/paper_result/summary_df.csv', index_col=0)
summary_df_pre = summary_df_pre.sort_values(by='score', ascending=False)

In [ ]:
cover_res = {}
eco_meta = meta_df.loc[eco_idx, :]

clade_num = 5
for key in ['啮齿目','真盲缺目','非洲兽目','翼手目','鲸目']:
    
    if key not in eco_meta['order_chinese_new'].values:
        cover_res[key] = np.zeros(len(eco_mutation))
        clade_num = 4
    else:
        idx = eco_meta['order_chinese_new'] == key 
        cover_res[key] = (eco_feature.loc[idx,:] == eco_mutation).sum(axis = 0) / idx.sum()
cover_df = pd.DataFrame(cover_res)
cover_df.loc[:,'NMI'] = nmi_list 
cover_df.loc[:,'Eco_Mode'] = eco_mutation 
cover_df.loc[:,'eco_cover'] = (cover_df.loc[:,['啮齿目','真盲缺目','非洲兽目','翼手目','鲸目']] > 0).sum(1)

    

In [53]:
summary_df_pre

,NMI,Eco_Mode,gene,非洲兽目,鲸目,啮齿目,真盲缺目,翼手目,eco_cover,eco_cover_score,score
MRO_68,6.940434e-01,M,MRO,0.0,0.8,1.0,1.0,1.0000,4,0.90,6.246390e-01
SLC26A5_496,5.719839e-01,M,SLC26A5,1.0,1.0,1.0,0.5,0.8750,5,1.00,5.719839e-01
ZC3H3_249,6.251362e-01,P,ZC3H3,1.0,0.9,0.0,0.5,0.8750,4,0.90,5.626226e-01
CWF19L1_403,6.808954e-01,I,CWF19L1,0.0,1.0,0.0,0.5,0.8750,3,0.75,5.106715e-01
CCDC92_3,5.628020e-01,A,CCDC92,1.0,0.9,1.0,0.0,0.6250,4,0.90,5.065218e-01
...,...,...,...,...,...,...,...,...,...,...,...
PKD1_10,2.285072e-06,L,PKD1,0.0,0.7,1.0,0.0,0.6875,3,0.75,1.713804e-06
DGCR2_111,5.281625e-07,G,DGCR2,1.0,0.6,0.0,0.5,0.5625,4,0.90,4.753462e-07
DGCR2_110,5.281625e-07,L,DGCR2,1.0,0.6,0.0,0.5,0.5625,4,0.90,4.753462e-07
DGCR2_109,5.281625e-07,F,DGCR2,1.0,0.6,0.0,0.5,0.5625,4,0.90,4.753462e-07


In [ ]:
### 获取eco mutation
eco_idx = meta_df['label'].values == 1 
eco_feature = feature_df.loc[eco_idx,:]
eco_mutation = eco_feature.mode(axis = 0).iloc[0,:].values  



In [30]:
eco_mutation

,MRO_68,SLC26A5_496,ZC3H3_249,CWF19L1_403,CCDC92_3,SLC26A5_307,GLB1_334,ANXA1_211,FOXK1_411,NT5C1A_26,...,ZNF638_533,HDAC6_841,POLRMT_1174,NACAD_467,COBLL1_442,ZNF646_1478,F5_1329,PDCD11_1481,TLR2_160,PDZD2_1746
0,M,M,P,I,A,S,V,K,S,H,...,-,K,T,-,A,L,-,R,S,Q
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,P,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
